## Here I will implement a learnable activation network. This combines elements of KANs and MLPs, treating the network as a standard MLP with simple edges, but learnable activation functions on the nodes rather than fixed ones. These appear better for classification tasks than KANs, though are considerably less interpretable.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [2]:
class PhyLAN(nn.Module):
    def __init__(self, LANshape, filters_per_unit, lr_decay=0.99, sigmoid_s=0.5):
        super().__init__()
        # Find number of layers
        self.Nlayers = len(LANshape)-1
        self.LANshape = LANshape
        # Create parameters for each layer
        self.filter_params = []
        self.weights = []
        for i in range(self.Nlayers):
            layer_weights = 2*(torch.rand((LANshape[i], LANshape[i+1]))-0.5)*(torch.sqrt(torch.tensor(6/(LANshape[i]+LANshape[i+1]))))
            layer_weights.requires_grad=True
            low_params = torch.linspace(-1, 1, filters_per_unit).unsqueeze(0).unsqueeze(-1)
            layer_params = torch.tile(low_params, (LANshape[i+1], 1, 2))
            noise = (torch.rand(layer_params.shape)-0.5)*0.01
            layer_params = layer_params + noise
            gains = torch.rand((LANshape[i+1], filters_per_unit, 1))/filters_per_unit
            params = torch.cat((gains, layer_params), dim=2)
            params.requires_grad=True
            self.filter_params.append(params)
            self.weights.append(layer_weights)
        # Initialise optimisers
        self.filter_optim = torch.optim.Adam(params=self.filter_params, lr=1e-5)
        self.weight_optim = torch.optim.Adam(params=self.weights, lr=1e-4)
        self.sched1 = torch.optim.lr_scheduler.ExponentialLR(self.filter_optim, 0.99)
        self.lossfn = nn.MSELoss()
        self.sigmoid_s = sigmoid_s
        
    def sig(self, x):
        return 1/(1+torch.exp(-x/self.sigmoid_s))
        
        
    # Redefined for torch & assuming trainable parameters
    def band_pass(self, Xin, params):
        # Bound parameters
        params= self.sig(params)
        # Encode inputs to frequency space
        freq = 10**(3.69+2*Xin.sigmoid()).unsqueeze(-1)
        # Allow bounded gains to be both positive and negative
        gain = 10*(params[:,:, 0].unsqueeze(0)-0.5)
        # Set cutoff frequencies from parameters
        fc_low = 10**(3.5+3*params[:,:, 1].unsqueeze(0))
        fc_high = 10**(3.5+3*params[:,:, 2].unsqueeze(0))
        # Convert cutoff frequency to product of resistance and capacitance
        RC_low = (2*torch.pi*fc_low)**-1
        RC_high = (2*torch.pi*fc_high)**-1
        # Calculate steady state response with respect to drive frequency
        Hout = gain*torch.abs((2j*torch.pi*freq*RC_low/(1+2j*torch.pi*freq*RC_low))*(1/(1+2j*torch.pi*freq*RC_high)))
        # Sum across units within each edge
        return Hout.sum(dim=-1)
    
    def forward(self, Xin):
        batch_size = len(Xin)
        # Initialise inputs to and outputs from each layer
        inputs = []
        outputs = []
        for i in range(self.Nlayers):
            inputs.append(torch.zeros((batch_size, self.LANshape[i])))
            outputs.append(torch.zeros((batch_size, self.LANshape[i+1])))
        inputs[0] = Xin
        # Pass through model
        for layer in range(self.Nlayers):
            weighted_inputs = torch.matmul(inputs[layer], self.weights[layer])
            outputs[layer][:, :] = self.band_pass(weighted_inputs, self.filter_params[layer])
            if layer < self.Nlayers-1:
                inputs[layer+1] = outputs[layer]
        return outputs[-1]
    
    def train(self, Xin, Yin):
        # Reset optimisers
        self.filter_optim.zero_grad()
        self.weight_optim.zero_grad()
        # Pass through Model
        pred = self.forward(Xin)
        # Calculate loss
        loss = self.lossfn(pred, Yin)
        # Backward pass
        loss.backward()
        # Update parameters
        self.filter_optim.step()
        self.weight_optim.step()
        return loss.item()

In [ ]:
if torch.cuda.is_available()==True:
    torch.set_default_tensor_type(torch.cuda.FloatTensor)
    device='cuda'
train_inputs = torch.load('Encoded train inputs 15.pt').to('cuda')
train_labels = torch.load('Training Targets15.pt').to('cuda')
test_inputs = torch.load('Encoded test inputs15.pt').to('cuda')
test_labels = torch.load('Test Targets15.pt').to('cuda')
train_inputs_np = train_inputs.cpu().detach().numpy()

def gen_samples(Xin, Yin, batch_size):
    indices = torch.randint(len(Xin), (batch_size,), device='cuda')
    Xsample = Xin[indices]
    Ysample = Yin[indices]
    return Xsample, Ysample

model = PhyLAN([15, 50, 50, 10], 6)
from tqdm import tqdm
for i in range(1000000):
    Xs, Ys = gen_samples(train_inputs, train_labels, 500)
    loss = model.train(Xs, Ys)
    if i%1000 == 0:
        #model.prune(20, 1000, threshold=0.1)
        print(loss)
        prediction = model.forward(test_inputs[:1000])
        correct = 0
        for i in range(1000):
            if torch.argmax(prediction[i])==torch.argmax(test_labels[i]):
                correct +=1
        accuracy = correct/1000
        print('Accuracy: ', accuracy)


/home/ian/anaconda3/lib/python3.9/site-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /opt/conda/conda-bld/pytorch_1720538616722/work/torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)
/tmp/ipykernel_56158/1150575053.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless the

0.30067893862724304
Accuracy:  0.105
0.16140110790729523
Accuracy:  0.115
0.13400621712207794
Accuracy:  0.115
0.11790674924850464
Accuracy:  0.115
0.1065313071012497
Accuracy:  0.115
0.09892376512289047
Accuracy:  0.095
0.09390538185834885
Accuracy:  0.095
0.09161096811294556
Accuracy:  0.095
0.0908133015036583
Accuracy:  0.095
0.09001736342906952
Accuracy:  0.095
0.08895493298768997
Accuracy:  0.158
0.08573327213525772
Accuracy:  0.237
0.07935973256826401
Accuracy:  0.327
0.07607171684503555
Accuracy:  0.414
0.06911087781190872
Accuracy:  0.454
0.06549407541751862
Accuracy:  0.488
0.06255938857793808
Accuracy:  0.514
0.05967959016561508
Accuracy:  0.578
0.053379546850919724
Accuracy:  0.632
0.050445448607206345
Accuracy:  0.669
0.04901149123907089
Accuracy:  0.699
0.049015436321496964
Accuracy:  0.714
0.04499455913901329
Accuracy:  0.722
0.04618004709482193
Accuracy:  0.738
0.0422893725335598
Accuracy:  0.74
0.04139384254813194
Accuracy:  0.749
0.04172607883810997
Accuracy:  0.751
0.

In [ ]:
weights = model.weights
for layer in weights: print(layer.size())